In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import numpy as np
import glob
import jax
import jax.numpy as jnp
import torch
from feature_functions import dino_feature_extractor
from model_utils import (toggle_prompt_embedding_shape,
                         expand_prompt_embedding,
    fitness_function,
    generate_images_from_latents,
    get_text_embeddings,
    plot_thumbnails
)
from diffusers import (
    UNet2DConditionModel,
    AutoencoderKL,
    LMSDiscreteScheduler
)
from qdax.core.containers.dns_repertoire import DNSRepertoire
from qdax.core.containers.mapelites_repertoire import compute_cvt_centroids, MapElitesRepertoire
from qdax.core.emitters.mutation_operators import isoline_variation
from qdax.core.emitters.standard_emitters import MixingEmitter
from qdax.core.map_elites import MAPElites

from transformers import (
    AutoImageProcessor,
    AutoModel,
    CLIPTextModel,
    CLIPTokenizerFast,
    CLIPModel
)
from matplotlib import pyplot as plt
%matplotlib inline
from PIL import Image
from functools import partial
from typing import Optional, Union, List, Callable

# Hyperparams

In [2]:
prompt = "an espresso machine"
test_images_dirs = ["test_repertoire_dataset/an_espresso_machine/gpt_crafted/images"]
descriptor_name = "dino-v2"
max_images = 5
project_desc = True

# Load Images

## Utils

In [3]:
def load_images_from_folder(folder, max_images=None):
    print("Loading images...")

    supported_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG"]
    image_paths = []
    for ext in supported_extensions:
        image_paths.extend(glob.glob(os.path.join(folder, ext)))

    if max_images:
        image_paths = image_paths[:max_images]

    images = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")  # Important: force RGB
                images.append(img.copy())  # .copy() keeps image after context closes
        except Exception as e:
            print(f"Error loading {path}: {e}")

    return images

In [4]:
test_images = []

for dir in test_images_dirs:
    test_images.extend(load_images_from_folder(dir, max_images=max_images))

print("Loaded {} images.".format(len(test_images)))
test_images = np.array(test_images)

Loading images...
Loaded 5 images.


# Descriptors

In [7]:
# project_desc = False
descriptor_name = "dino-v3"

if descriptor_name == "dino-v2":
    dino_model = AutoModel.from_pretrained("facebook/dinov2-base").eval().to("cuda")
    dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
    
    if project_desc == True:
        projection_matrix = torch.randn(2, 768, device="cuda")
    else:
        projection_matrix = None
        
    descriptor_fn =  partial(
    dino_feature_extractor,
        processor=dino_processor,
        model=dino_model,
        projection_matrix=projection_matrix,
        device="cuda"
    )

elif descriptor_name == "dino-v3":
    dino_model = AutoModel.from_pretrained("facebook/dinov3-convnext-tiny-pretrain-lvd1689m").eval().to("cuda")
    dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov3-convnext-tiny-pretrain-lvd1689m")
    
    if project_desc == True:
        projection_matrix = torch.randn(2, 1024, device="cuda")
    else:
        projection_matrix = None
        
    descriptor_fn =  partial(
    dino_feature_extractor,
        processor=dino_processor,
        model=dino_model,
        projection_matrix=torch.randn(2, 1024, device="cuda"),
        device="cuda"
    )

descs = [descriptor_fn(img) for img in test_images]
print("Computed descriptors shape:", np.array(descs).shape)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-convnext-tiny-pretrain-lvd1689m.
401 Client Error. (Request ID: Root=1-69009b83-37f12b31265d9599760f278e;9fe9e3cb-c8cf-4f01-9f8b-20c382b1360c)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-convnext-tiny-pretrain-lvd1689m/resolve/main/config.json.
Access to model facebook/dinov3-convnext-tiny-pretrain-lvd1689m is restricted. You must have access to it and be authenticated to access it. Please log in.

# Repertoires

# Scoring Function and Set Up

In [ ]:

def scoring_fn(latents,
    prompt_embedding,
    key,
    prompt,
    clip_model,
    clip_tokenizer,
    generate_images_from_latents_fun,
    descriptor_functions: List[Callable],
    descriptor_type: str,
):
    # Generate images
    generated_images = generate_images_from_latents_fun(latents, prompt_embedding)
            
    fitnesses = fitness_function(generated_images=generated_images,
        prompt=prompt,
        clip_model=clip_model,
        clip_tokenizer=clip_tokenizer,
    )
    
    # Convert generated images to numpy
    generated_images_np = np.array(generated_images.cpu())
            
    descriptors = []
    
    for d_fun in descriptor_functions:
        descriptor = jnp.array([d_fun(image) for image in generated_images_np])           
        descriptors.append(descriptor)
    
    if descriptor_type == "dino":
        descriptors = jnp.atleast_2d(jnp.array(descriptors).squeeze()) # Shape: (batch_size, num_descriptors)     
    else:
        descriptors = jnp.stack(descriptors, axis=-1)
        
    return fitnesses, descriptors, {}


def score_batch(
    latents,
    embeddings,
    prompt,
    key,
    max_scoring_batch=4,  
):
    generate_images_from_latents_partial = partial(generate_images_from_latents,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    )
    
    scoring_function = partial(scoring_fn,
        prompt=prompt,
        clip_model=clip_model,
        clip_tokenizer=tokenizer,
        generate_images_from_latents_fun=generate_images_from_latents_partial,
        descriptor_functions=[dino_fun],
        descriptor_type=["dino"],
    )
    # Compute the MAP-Elites centroids
    key, subkey = jax.random.split(key)

    # --- Batched scoring to avoid CUDA OOM ---
    # Make per-chunk RNG keys (reusing 'key' now that 'subkey' went to centroids)
    
    n = len(latents)
    num_chunks = (n + max_scoring_batch - 1) // max_scoring_batch
    chunk_keys = jax.random.split(key, num_chunks)

    fitnesses_list = []
    descriptors_list = []

    for ch in range(num_chunks):
        i = ch * max_scoring_batch
        j = min((ch + 1) * max_scoring_batch, n)
        b = j - i

        # Slice this chunk
        lat_ch = latents[i:j]           # (b, 4, 64, 64)
        geno_ch = embeddings[i:j]        # (b, 2, 77, 1024) or your toggled shape

        # --- FIX 1: convert to NumPy before JAX ---
        if isinstance(geno_ch, torch.Tensor):
            geno_ch_np = geno_ch.detach().cpu().numpy()
        else:
            geno_ch_np = np.asarray(geno_ch)
            
        test_genotypes = toggle_prompt_embedding_shape(jnp.array(geno_ch))

        # Run scoring on the chunk only
        fit_ch, desc_ch, _ = scoring_function(lat_ch, test_genotypes, chunk_keys[ch])
        
        # --- FIX 2: normalize singleton chunk shapes ---
        fit_ch = jnp.asarray(fit_ch)
        if fit_ch.ndim == 0:      # () -> (1,)
            fit_ch = fit_ch[None]
        else:
            fit_ch = jnp.squeeze(fit_ch)
            if fit_ch.ndim > 1:   # collapse extras if any
                fit_ch = fit_ch.reshape(b, -1).mean(axis=1)  # or pick [:,0] if that's your semantics
                
        desc_ch = jnp.asarray(desc_ch)
        if desc_ch.ndim == 1:     # (D,) -> (1, D) when b==1
            desc_ch = desc_ch[None, :]
        else:
            desc_ch = jnp.squeeze(desc_ch)
        # final shape checks
        if fit_ch.shape[0] != b:
            raise ValueError(f"fitnesses: expected length {b}, got {fit_ch.shape}")
        if desc_ch.shape[0] != b:
            # try last-resort reshape if it’s flat of size b*D
            if desc_ch.ndim == 1 and desc_ch.size % b == 0:
                desc_ch = desc_ch.reshape(b, -1)
            if desc_ch.shape[0] != b:
                raise ValueError(f"descriptors: expected leading dim {b}, got {desc_ch.shape}")
        
        fitnesses_list.append(fit_ch)
        descriptors_list.append(desc_ch)

        # Proactively free GPU memory between chunks
        torch.cuda.empty_cache()

    # Concatenate results back to full batch
    fitnesses = jnp.concatenate(fitnesses_list, axis=0)
    descriptors = jnp.concatenate(descriptors_list, axis=0)
    print("Descriptors:", descriptors)
    print("Fitnesses:", fitnesses)
    
    return fitnesses, descriptors


def test_added_to_repertoire(genotypes,
    fitnesses,
    descriptors,
    max_dns_size=8, 
    num_mapelites_centroids=8,
    num_descriptors=2,                     
):
    
    centroids = compute_cvt_centroids(
        num_descriptors=num_descriptors,
        num_init_cvt_samples=50000,
        num_centroids=num_mapelites_centroids,
        minval=-50.0,
        maxval=50.0,
        key=subkey,
    )

    variation_fn = partial(
        isoline_variation,
        iso_sigma=0.05,
        line_sigma=0.5
    )
            
    mixing_emitter = MixingEmitter(
        mutation_fn=None,
        variation_fn=variation_fn,
        variation_percentage=1.0,
        batch_size=4
    )
    
    def dummy_metrics_fn(repertoire):
        return {}
    
    def dummy_scoring_fn(genotypes, key):
        return jnp.zeros(len(genotypes)), jnp.zeros((len(genotypes), 2)), {}
    
    # Instantiate MAP-Elites
    dns = MAPElites(
        scoring_function=dummy_scoring_fn,
        emitter=mixing_emitter,
        metrics_function=dummy_metrics_fn,
        repertoire_init=partial(DNSRepertoire.init, max_size=max_dns_size)
    )    
    
    # Prepare initial population (JAX)
    dns_repertoire, _, is_offspring_added = dns.init_ask_tell(
        genotypes=jnp.array(genotypes, dtype=jnp.float32),
        fitnesses=fitnesses,
        descriptors=descriptors,
        centroids=centroids,
        key=key,
        extra_scores=None,
    )

    # Instantiate MAP-Elites
    map_elites = MAPElites(
        scoring_function=dummy_scoring_fn,
        emitter=mixing_emitter,
        metrics_function=dummy_metrics_fn,
        repertoire_init=partial(MapElitesRepertoire.init, 
            centroids=centroids,
            extra_scores={})
    )    

    map_elites_repertoire, _, is_me_offspring_added = map_elites.init_ask_tell(
        genotypes=jnp.array(genotypes, dtype=jnp.float32),
        fitnesses=fitnesses,
        descriptors=descriptors,
        centroids=centroids,
        key=key,
        extra_scores=None,
    )
    
    return dns_repertoire, map_elites_repertoire, is_offspring_added, is_me_offspring_added